**import necessary types**

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType,IntegerType,DateType,FloatType,TimestampType

catalog_name='ecommerce'

**read bronze table**

In [0]:
df_bronze=spark.table(f"{catalog_name}.bronze.brz_brands")
df_bronze.show(4)

**basic cleaning**-trimming

In [0]:
df_silver=df_bronze.withColumn('brand_name',F.trim(F.col('brand_name')))
df_silver.show(10)

some unwanted characters in brand_code column- use regex

In [0]:
df_silver=df_silver.withColumn('brand_code',F.regexp_replace(F.col('brand_code'),r'[^A-Za-z0-9]',''))
df_silver.show(11)

In [0]:
df_silver.select('category_code').distinct().show()

replace similar naming categories- bks, books...

In [0]:
#Anomalies dictionary
anomalies={
    "GROCERY":"GRCY",
    "BOOKS":"BKS",
    "TOYS":"TOY"
}

df_silver=df_silver.replace(anomalies,subset='category_code')

df_silver.select('category_code').distinct().show()

**write raw data to silver layer**

In [0]:
df_silver.write.format("delta")\
  .mode("overwrite")\
    .option("mergeSchema","true")\
      .saveAsTable(f"{catalog_name}.silver.slv_brands")

**Category**

In [0]:
df_brz_cat=spark.table(f"{catalog_name}.bronze.brz_category")
df_brz_cat.show(15)

identify and remove duplicates

In [0]:
df_duplicates_cat=df_brz_cat.groupBy('category_code').count().filter(F.col("count")>1)
display(df_duplicates_cat)

In [0]:
df_silver_cat=df_brz_cat.dropDuplicates(['category_code'])
df_silver_cat.show(15)

In [0]:
df_silver_cat=df_silver_cat.withColumn('category_code',F.upper(F.col('category_code')))
df_silver_cat.show(15)

In [0]:
df_silver_cat.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema","true")\
        .saveAsTable(f"{catalog_name}.silver.slv_category")
        

**Products**

In [0]:
df_brz_prd=spark.table(f"{catalog_name}.bronze.brz_products")
df_brz_prd.show(10)

#get row and column count

row_count, column_count=df_brz_prd.count(),len(df_brz_prd.columns)

print(f"row count-{row_count},column count-{column_count}")


check weight 'g'

In [0]:
df_brz_prd.select('weight_grams').show(5,truncate=False)

In [0]:
#replace g 

df_silver_prd=df_brz_prd.withColumn('weight_grams',F.regexp_replace(F.col('weight_grams'),'g','').cast(IntegerType()))

df_silver_prd.select('weight_grams').show(5,truncate=False)

In [0]:
df_silver_prd=df_silver_prd.withColumn('category_code',F.upper(F.col('category_code'))).withColumn(
    'brand_code',F.upper(F.col('brand_code')))

df_silver_prd.show(10)

spelling

In [0]:
df_silver_prd.select('material').distinct().show()

In [0]:
df_silver_prd=df_silver_prd.withColumn(
    'material',
    F.when(F.col('material')=="Coton","Cotton")
    .when(F.col("material")=="Ruber","Rubber")
    .when(F.col("material")=="Alumium","Aluminium")
    .otherwise(F.col("material"))
    )

df_silver_prd.select("material").distinct().show()

negative values in rating count

In [0]:
df_silver_prd.filter(F.col("rating_count")<0).select("rating_count").show(5)

In [0]:
df_silver_prd=df_silver_prd.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(),F.abs(F.col("rating_count")))
    .otherwise(F.lit(0))
)

#check final cleaned data

df_silver_prd.select(
    "weight_grams",
    "length_cm",
    "category_code",
    "brand_code",
    "material",
    "rating_count"
).show(15,truncate=False)

In [0]:
df_silver_prd.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema","true")\
        .saveAsTable(f"{catalog_name}.silver.slv_products")


**Customers**

In [0]:
df_brz_cust=spark.read.table(f"{catalog_name}.bronze.brz_customers")

#get row column count
row_cnt,col_cnt=df_brz_cust.count(),len(df_brz_cust.columns)

print(f" row count: {row_cnt}, column count: {col_cnt}")

df_brz_cust.show(10)

Handle null values in customer_id column

In [0]:
null_count=df_brz_cust.filter(F.col("customer_id").isNull()).count()
null_count

there are 300 null values in customer id column- display some of them

In [0]:
df_brz_cust.filter(F.col("customer_id").isNull()).show(4)

drop null value columns

In [0]:
df_silver_cust=df_brz_cust.dropna(subset=["customer_id"])

row_countsilver=df_silver_cust.count()

print(f"row count after dropping null values is {row_countsilver}")

handle null values in phone number

In [0]:
nul_count=df_silver_cust.filter(F.col("phone").isNull()).count()
nul_count

df_silver_cust.filter(F.col("phone").isNull()).show(5)


In [0]:
df_silver_cust=df_silver_cust.fillna("Not Available",subset=["phone"])

#since it's not a string column- this doesn't workout
#cast it to string

df_silver_cust=df_silver_cust.withColumn(
    "phone",
    F.when(F.col("phone").isNull(),"Not Available")
    .otherwise(F.col("phone").cast("String"))
    )

#df_silver_cust.printSchema()

df_silver_cust.filter(F.col("phone").isNull()).show()

In [0]:
df_silver_cust.write.format("delta").mode("overwrite").option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.silver.slv_customers")

**Date**

In [0]:
df_brz_dte=spark.read.table(f"{catalog_name}.bronze.brz_date")

df_brz_dte.show(10)

rc,c_c=df_brz_dte.count(),len(df_brz_dte.columns)

print(f"row count-{rc} column count-{c_c}")


In [0]:
df_brz_dte.printSchema()

Convert string to date

In [0]:
from pyspark.sql.functions import to_date

df_silver_dte=df_brz_dte.withColumn("date",to_date(df_brz_dte["date"], "dd-MM-yyyy"))

df_silver_dte.printSchema()

df_silver_dte.show(5)

remove duplicates

In [0]:
#find dup
dup=df_silver_dte.groupBy("date").count().filter("count>1")

print("total duplicate rows",dup.count())

display(dup)

In [0]:
df_silver_dte=df_silver_dte.dropDuplicates(["date"])

#check row count

rcc=df_silver_dte.count()

print("rows after removing duplicates ",rcc)


modify day name column

In [0]:
df_silver_dte=df_silver_dte.withColumn("day_name",F.initcap(F.col("day_name")))
df_silver_dte.show(5)

convert week of year

In [0]:
df_silver_dte=df_silver_dte.withColumn("week_of_year",F.abs(F.col("week_of_year")))

df_silver_dte.show(4)

quarter, week of year enhancements

In [0]:
#clean column before changing- converting to int

df_silver_dte=df_silver_dte.withColumn("week_of_year",F.expr("try_cast(week_of_year as int)"))

# F.expr- for sql qu inside pyspark
# try_cast- if value is null, it doesn't give error

df_silver_dte=df_silver_dte.withColumn("quarter",F.concat_ws("-",F.concat(F.lit("Q"),F.col("quarter")),F.col("year")))

#F.concat_ws- for concat with separator- in this case "-"
#F.concat-for concat without separator
#F.lit-for literal value

df_silver_dte=df_silver_dte.withColumn("week_of_year",F.concat_ws("-",F.lit("week"),F.col("week_of_year"),F.col("year")))

df_silver_dte.show(10)

Rename columns

In [0]:
df_silver_dte=df_silver_dte.withColumnRenamed("week_of_year","week")

df_silver_dte.show(2)

In [0]:
df_silver_dte.write.format("delta").mode("overwrite").option("mergeSchema","true").\
    saveAsTable(f"{catalog_name}.silver.slv_date")